
# Cross-validation strategies

The previous notebooks introduced how to evaluate a model and how to create a
specific preprocessing pipeline depending on the last model.

In this notebook, we will look more closely at some details of the cross-validation
strategies and at some of the pitfalls that we can encounter.

Let's take the iris dataset and evaluate a logistic regression model.


## Environment setup

Run this cell in JupyterLite. A local installation already provides these packages.

In [ ]:
%pip install -q "scikit-learn~=1.9.1" "skrub~=0.11.0" pandas matplotlib

In [ ]:
from sklearn.datasets import load_iris

df, target = load_iris(as_frame=True, return_X_y=True)

In [ ]:
df

In [ ]:
target

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logistic_regression = make_pipeline(StandardScaler(), LogisticRegression())
logistic_regression


### Exercise

Evaluate the previous {py:class}`~sklearn.linear_model.LogisticRegression` model using a 3-fold cross-validation
(i.e. {py:class}`sklearn.model_selection.KFold`). What do you observe?


We observe that the training score is always zero, which is really surprising. We can
check the target to understand why.


We can use a {py:class}`~sklearn.model_selection.StratifiedKFold` object to ensure that the class distribution is
preserved in each fold. A side effect will be that all classes will be present in the
training set and testing set.

In [ ]:
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=3)
cv_results = cross_validate(logistic_regression, df, target, cv=cv, return_train_score=True)
cv_results = pd.DataFrame(cv_results)
cv_results[["train_score", "test_score"]]

In [ ]:
for cv_fold_idx, (train_indices, test_indices) in enumerate(cv.split(df, target)):
    print(f"Fold {cv_fold_idx}:\n")
    print(f"Class counts on the train set:\n" f"{target.iloc[train_indices].value_counts()}")
    print(f"Class counts on the test set:\n" f"{target.iloc[test_indices].value_counts()}")
    print()


This is particularly useful when we have imbalanced classes. Let's check the class
distribution of the breast cancer dataset.

In [ ]:
from sklearn.datasets import load_breast_cancer

df, target = load_breast_cancer(as_frame=True, return_X_y=True)

In [ ]:
target.value_counts(normalize=True)


Here, we see that the proportion of the two classes is not equal. We can check the
class distribution in each fold using a {py:class}`~sklearn.model_selection.KFold` object.

In [ ]:
cv = KFold(n_splits=3, shuffle=True, random_state=0)
for cv_fold_idx, (train_indices, test_indices) in enumerate(cv.split(df, target)):
    print(f"Fold {cv_fold_idx}:\n")
    print(
        "Class counts on the train set:\n"
        f"{target.iloc[train_indices].value_counts(normalize=True)}\n"
    )
    print(f"Class counts on the test set:\n" f"{target.iloc[test_indices].value_counts(True)}")
    print()


We observe that the class distribution is not preserved in each fold. We can use a
{py:class}`~sklearn.model_selection.StratifiedKFold` object to ensure that the class distribution is preserved in each
fold.

In [ ]:
cv = StratifiedKFold(n_splits=3)
for cv_fold_idx, (train_indices, test_indices) in enumerate(cv.split(df, target)):
    print(f"Fold {cv_fold_idx}:\n")
    print(
        "Class counts on the train set:\n"
        f"{target.iloc[train_indices].value_counts(normalize=True)}\n"
    )
    print(f"Class counts on the test set:\n" f"{target.iloc[test_indices].value_counts(True)}")
    print()


Now, let's check the documentation of the {py:func}`~sklearn.model_selection.cross_validate` function to see if this
function already provides a way to stratify the data.

In [ ]:
help(cross_validate)


Now, we will look at the notion of `groups` in cross-validation. We will use the
digits dataset and group the samples by writer.

In [ ]:
from sklearn.datasets import load_digits

df, target = load_digits(return_X_y=True)


We create a simple model that is a logistic regression model with a scaling of the
data.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

logistic_regression = make_pipeline(MinMaxScaler(), LogisticRegression())


### Exercise

Evaluate the model and compare two strategies:
- using a {py:class}`~sklearn.model_selection.KFold` object with 13 splits without shuffling the data;
- using a {py:class}`~sklearn.model_selection.KFold` object with 13 splits with shuffling the data.

What do you observe? What are the causes of the differences?


Here, we provide a `groups` array that gives the writer ID for each sample.

In [ ]:
from itertools import count

import numpy as np

# Define the lower and upper bounds of sample indices
# for each writer.
writer_boundaries = [
    0,
    130,
    256,
    386,
    516,
    646,
    776,
    915,
    1029,
    1157,
    1287,
    1415,
    1545,
    1667,
    1797,
]
groups = np.zeros_like(target)
lower_bounds = writer_boundaries[:-1]
upper_bounds = writer_boundaries[1:]

for group_id, lb, up in zip(count(), lower_bounds, upper_bounds):
    groups[lb:up] = group_id

In [ ]:
import matplotlib.pyplot as plt

plt.plot(groups)
plt.yticks(np.unique(groups))
plt.xticks(writer_boundaries, rotation=90)
plt.xlabel("Target index")
plt.ylabel("Writer index")
_ = plt.title("Underlying writer groups present in the target")


We can use this information to properly evaluate our model. We need to use the
{py:class}`~sklearn.model_selection.GroupKFold` object and pass the `groups` parameter to the {py:func}`~sklearn.model_selection.cross_validate` function.

In [ ]:
from sklearn.model_selection import GroupKFold

cv = GroupKFold(n_splits=13)
cv_results = cross_validate(logistic_regression, df, target, groups=groups, cv=cv)
print(
    f"Mean test score: {cv_results['test_score'].mean():.3f} +/- "
    f"{cv_results['test_score'].std():.3f}"
)


We observe that the mean test score is even lower but certainly closer to the true
performance of the model.